# 01 - Exploratory Data Analysis: Bank Churn

Notebook nay thuc hien EDA chi tiet cho bai toan du doan churn khach hang ngan hang.

Muc tieu:
- Hieu cau truc du lieu, kieu du lieu va chat luong du lieu.
- Phan tich phan phoi target `exit`.
- So sanh nhom churn va non-churn theo numeric, categorical va date features.
- Tim cac bien co tin hieu manh cho model CatBoost.
- Dua ra insight va cac luu y cho train/predict/app.

## Nhan xet nhanh tu du lieu hien tai

- Tong so ban ghi: **80,000**.
- Tong so cot: **26**.
- Target: `exit`.
- Ty le churn: **18.00%** tuong ung **14,400** khach hang churn va **65,600** khach hang khong churn.
- Missing values: **0**.
- Duplicate rows: **0**.
- `customer_segment` co tin hieu rat manh: `Mass` churn khoang **39.48%**, `Priority` gan nhu khong churn.
- `risk_segment=Medium` churn khoang **58.87%**, cao hon nhieu so voi `Low`.
- `digital_behavior=offline` churn khoang **21.14%**, cao hon `mobile` khoang **6.50%**.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

sns.set_theme(style='whitegrid', palette='Set2')

DATA_PATH = Path('../data/raw/bank_churn_dataset.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/bank_churn_dataset.csv')

df = pd.read_csv(DATA_PATH)
df.head()

## 1. Tong quan dataset

In [ ]:
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
display(df.head())
display(df.tail())

In [ ]:
overview = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_pct': df.isna().mean() * 100,
    'n_unique': df.nunique(dropna=False),
})
display(overview.sort_values(['missing_count', 'n_unique'], ascending=[False, False]))

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate id:', df['id'].duplicated().sum() if 'id' in df.columns else 'No id column')
print('Memory usage MB:', round(df.memory_usage(deep=True).sum() / 1024**2, 2))

## 2. Phan tich target churn

In [ ]:
target_col = 'exit'
target_summary = df[target_col].value_counts(dropna=False).rename_axis('exit').reset_index(name='count')
target_summary['pct'] = target_summary['count'] / len(df) * 100
display(target_summary)

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x=target_col, ax=ax)
ax.set_title('Target distribution: exit')
ax.set_xlabel('Churn')
ax.set_ylabel('Number of customers')
for container in ax.containers:
    ax.bar_label(container, fmt='%d')
plt.show()

Nhan xet: dataset bi lech lop vua phai, churn chiem khoang 18%. Khi train model nen theo doi recall/F1/ROC-AUC thay vi chi accuracy. CatBoost trong project dang dung `auto_class_weights='Balanced'` de xu ly imbalance.

## 3. Numeric features

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
id_like_cols = ['id']
numeric_feature_cols = [c for c in numeric_cols if c not in id_like_cols]

display(df[numeric_feature_cols].describe().T)

In [ ]:
num_by_target = df.groupby(target_col)[numeric_feature_cols].agg(['mean', 'median', 'std']).T
display(num_by_target)

In [ ]:
plot_cols = ['credit_sco', 'age', 'balance', 'monthly_ir', 'tenure_ye', 'nums_card', 'nums_service', 'last_transaction_month', 'engagement_score', 'risk_score']
plot_cols = [c for c in plot_cols if c in df.columns]

n_cols = 2
n_rows = int(np.ceil(len(plot_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, plot_cols):
    sns.histplot(data=df, x=col, hue=target_col, bins=40, kde=False, stat='density', common_norm=False, ax=ax)
    ax.set_title(f'Distribution of {col} by churn')

for ax in axes[len(plot_cols):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
outlier_rows = []
for col in numeric_feature_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_rows.append({'column': col, 'lower': lower, 'upper': upper, 'outlier_count': outlier_count, 'outlier_pct': outlier_count / len(df) * 100})

outlier_report = pd.DataFrame(outlier_rows).sort_values('outlier_pct', ascending=False)
display(outlier_report)

## 4. Categorical features

In [ ]:
categorical_cols = df.select_dtypes(include=['object', 'string', 'bool']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in ['full_name']]

cat_overview = pd.DataFrame({
    'column': categorical_cols,
    'n_unique': [df[c].nunique(dropna=False) for c in categorical_cols],
    'top_value': [df[c].value_counts(dropna=False).index[0] for c in categorical_cols],
    'top_count': [df[c].value_counts(dropna=False).iloc[0] for c in categorical_cols],
})
display(cat_overview.sort_values('n_unique', ascending=False))

In [ ]:
def churn_rate_table(data: pd.DataFrame, col: str, min_count: int = 30) -> pd.DataFrame:
    result = (
        data.groupby(col)[target_col]
        .agg(count='count', churn_count='sum', churn_rate='mean')
        .reset_index()
    )
    result['churn_rate_pct'] = result['churn_rate'] * 100
    result = result[result['count'] >= min_count]
    return result.sort_values('churn_rate', ascending=False)

key_cat_cols = ['gender', 'occupation', 'origin_province', 'married', 'active_member', 'customer_segment', 'loyalty_level', 'digital_behavior', 'risk_segment', 'cluster_group']
key_cat_cols = [c for c in key_cat_cols if c in df.columns]

for col in key_cat_cols:
    print(f'==== {col} ====')
    display(churn_rate_table(df, col).head(20))

In [ ]:
plot_cat_cols = ['customer_segment', 'risk_segment', 'digital_behavior', 'loyalty_level', 'occupation', 'origin_province']
plot_cat_cols = [c for c in plot_cat_cols if c in df.columns]

fig, axes = plt.subplots(len(plot_cat_cols), 1, figsize=(12, 4 * len(plot_cat_cols)))
if len(plot_cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, plot_cat_cols):
    table = churn_rate_table(df, col).head(15).sort_values('churn_rate_pct')
    sns.barplot(data=table, x='churn_rate_pct', y=col, ax=ax)
    ax.set_title(f'Churn rate by {col}')
    ax.set_xlabel('Churn rate (%)')
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

Nhan xet quan trong:
- `risk_segment` va `customer_segment` co kha nang tach nhom churn rat ro.
- `digital_behavior=offline` co churn rate cao hon nhieu so voi `mobile`.
- Mot so cot co cardinality cao nhu `address`, `origin_province`, `occupation`; CatBoost xu ly categorical truc tiep nen khong can one-hot trong pipeline hien tai.

## 5. Date features

In [ ]:
date_cols = ['last_active_date', 'created_date']
for col in date_cols:
    if col in df.columns:
        df[col + '_parsed'] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

date_quality = []
for col in date_cols:
    parsed_col = col + '_parsed'
    if parsed_col in df.columns:
        date_quality.append({
            'column': col,
            'invalid_count': df[parsed_col].isna().sum(),
            'min_date': df[parsed_col].min(),
            'max_date': df[parsed_col].max(),
        })

display(pd.DataFrame(date_quality))

In [ ]:
if 'last_active_date_parsed' in df.columns:
    monthly_churn = (
        df.assign(last_active_month=df['last_active_date_parsed'].dt.to_period('M').astype(str))
        .groupby('last_active_month')[target_col]
        .agg(count='count', churn_rate='mean')
        .reset_index()
        .sort_values('last_active_month')
    )
    monthly_churn['churn_rate_pct'] = monthly_churn['churn_rate'] * 100
    display(monthly_churn.tail(24))

    fig, ax1 = plt.subplots(figsize=(14, 5))
    sns.lineplot(data=monthly_churn, x='last_active_month', y='churn_rate_pct', marker='o', ax=ax1)
    ax1.set_title('Monthly churn rate by last active month')
    ax1.set_xlabel('Last active month')
    ax1.set_ylabel('Churn rate (%)')
    ax1.tick_params(axis='x', rotation=60)
    plt.tight_layout()
    plt.show()

Luu y: trong pipeline hien tai, date columns dang duoc dua vao CatBoost nhu categorical raw string, dung theo yeu cau khong feature engineering. Neu muon nang model sau nay, co the tao feature nhu recency, tenure days, active month, created month.

## 6. Correlation va moi quan he numeric

In [ ]:
corr_df = df[numeric_feature_cols + [target_col]].copy()
corr_df[target_col] = corr_df[target_col].astype(int)
corr = corr_df.corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='RdBu_r', center=0, annot=False, linewidths=0.5)
plt.title('Correlation heatmap')
plt.show()

target_corr = corr[target_col].drop(target_col).sort_values(key=lambda s: s.abs(), ascending=False)
display(target_corr.to_frame('corr_with_exit'))

Correlation chi bat duoc quan he tuyen tinh cua numeric features. Voi CatBoost, categorical va nonlinear interaction van co the dong gop manh ngay ca khi correlation numeric thap.

## 7. Phan tich theo risk va segment

In [ ]:
if {'customer_segment', 'risk_segment'}.issubset(df.columns):
    segment_risk = (
        df.groupby(['customer_segment', 'risk_segment'])[target_col]
        .agg(count='count', churn_count='sum', churn_rate='mean')
        .reset_index()
    )
    segment_risk['churn_rate_pct'] = segment_risk['churn_rate'] * 100
    display(segment_risk.sort_values('churn_rate', ascending=False))

    pivot = segment_risk.pivot(index='customer_segment', columns='risk_segment', values='churn_rate_pct')
    plt.figure(figsize=(8, 4))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Reds')
    plt.title('Churn rate (%) by customer segment and risk segment')
    plt.show()

In [ ]:
if {'customer_segment', 'digital_behavior'}.issubset(df.columns):
    segment_digital = (
        df.groupby(['customer_segment', 'digital_behavior'])[target_col]
        .agg(count='count', churn_count='sum', churn_rate='mean')
        .reset_index()
    )
    segment_digital['churn_rate_pct'] = segment_digital['churn_rate'] * 100
    display(segment_digital.sort_values('churn_rate', ascending=False))

    plt.figure(figsize=(10, 5))
    sns.barplot(data=segment_digital, x='customer_segment', y='churn_rate_pct', hue='digital_behavior')
    plt.title('Churn rate by segment and digital behavior')
    plt.ylabel('Churn rate (%)')
    plt.xlabel('Customer segment')
    plt.show()

## 8. Train/Test files da split

In [ ]:
train_path = Path('../data/train.csv')
test_path = Path('../data/test.csv')

if train_path.exists() and test_path.exists():
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    print('train.csv shape:', train_df.shape)
    print('test.csv shape:', test_df.shape)
    print('Target in train:', target_col in train_df.columns)
    print('Target in test:', target_col in test_df.columns)
    display(train_df[target_col].value_counts(normalize=True).rename('train_target_rate'))
else:
    print('Chua thay data/train.csv hoac data/test.csv')

## 9. Ket luan EDA

Cac insight chinh:
- Dataset sach: khong missing, khong duplicate, id unique.
- Target imbalance o muc vua phai: churn 18%, non-churn 82%.
- `risk_segment`, `risk_score`, `customer_segment`, `digital_behavior`, `engagement_score`, `last_transaction_month` co kha nang la nhom feature quan trong.
- Nhom `Mass`, `risk_segment=Medium`, va `offline` co churn rate cao hon ro ret.
- Dung CatBoost la phu hop vi du lieu co nhieu categorical feature, bao gom cot cardinality cao.

Khuyen nghi tiep theo:
- Khi danh gia model, uu tien ROC-AUC, recall, precision, F1 va confusion matrix.
- Trong app, nen cho phep dieu chinh threshold vi bai toan churn thuong can toi uu recall hoac precision theo chien dich kinh doanh.
- Neu nang cap model sau nay, co the them feature engineering tu ngay thang va hanh vi giao dich, nhung pipeline hien tai dang giu dung yeu cau khong feature engineering.